In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from scipy.spatial import distance
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
# Read the data from txt files to pandas DataFrame
# Data read as numpy.float64 scaled into the range [-1.0:1.0]
isolet1234 = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/ISOLET/isolet/isolet1+2+3+4.data', sep=',\s+', header=None)
isolet5 = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/ISOLET/isolet/isolet5.data', header=None)

<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_1492/3479077733.py:3: SyntaxWarning: invalid escape sequence '\s'
  isolet1234 = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/ISOLET/isolet/isolet1+2+3+4.data', sep=',\s+', header=None)
/tmp/ipykernel_1492/3479077733.py:3: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  isolet1234 = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/ISOLET/isolet/isolet1+2+3+4.data', sep=',\s+', header=None)


In [ ]:
# Convert pandas DataFrame PyTorch Tensor
#   Columns [0:616]: features
#   Columns [617]: label
train_samples_float = torch.tensor(isolet1234.values)[:, :617]
train_labels = torch.tensor(isolet1234.values).to(torch.uint8)[:, 617] - 1 # -1: Put the labels in range [0:25] instead the original [1:26]

test_samples_float = torch.tensor(isolet5.values)[:, :617]
test_labels = torch.tensor(isolet5.values).to(torch.uint8)[:, 617] - 1 # -1: Put the labels in range [0:25] instead the original [1:26]

In [ ]:
# Convert dataset to uint8
# Data read as numpy.float64 scaled into the range [-1.0:1.0]

scale_factor = 2
offset = scale_factor
train_samples = (train_samples_float * scale_factor).round().to(torch.uint8) + offset
test_samples = (test_samples_float * scale_factor).round().to(torch.uint8) + offset

# Create a TensorDataset and DataLoader
#dataloader = torch.utils.data.DataLoader(TensorDataset(train_samples), batch_size=2, shuffle=True)

In [ ]:
# Constants
DIMENSIONS = 8192
FEATURE_LEVELS = 5
FEATURES = 617
CLASSES = 26

In [ ]:
# Key-value encoding
def Encode(dimensions, sample, levels, indexes):
    encoded = torch.zeros(dimensions, dtype=torch.int32)

    for i in range(sample.shape[0]):
        encoded += levels[sample[i].item()] ^ indexes[i]

    # Addition with a majority vote threshold
    sample_size = sample.shape[0]
    encoded_bin = torch.where(encoded > (sample_size // 2), 1, 0)

    return encoded_bin.to(dtype=torch.int8), encoded

In [ ]:
# Seed generation

def BinaryTensor(n, size):

    bin = torch.zeros(size, dtype=torch.uint8)

    for i in range(size):
        bin[size - 1 - i] = n % 2
        n //= 2

    return bin

def ISOLET_HVBit(t):
    a = torch.zeros_like(t)

    a[0] = t[0] ^ t[1]
    a[1] = t[2] ^ t[3]
    a[2] = t[4] ^ t[5]
    a[3] = t[6] ^ t[7]
    a[4] = t[8] ^ t[9]
    a[5] = t[10] ^ t[11]
    a[6] = t[12] ^ t[13]

    return a[0] ^ a[1] ^ a[2] ^ a[3] ^ a[4] ^ a[6]

gen_seed = torch.zeros(DIMENSIONS, dtype=torch.uint8)

size = 16

for i in range(DIMENSIONS):
    gen_seed[i] = ISOLET_HVBit(BinaryTensor(i, size).roll(-(i % 16)))

In [ ]:
# GENERATED SEED

# Create hypervectors table: each value has an hypervector
level_hvs_bin = torch.zeros([FEATURE_LEVELS, DIMENSIONS], dtype=torch.uint8)
seed = gen_seed

for level in range(FEATURE_LEVELS):
    level_hvs_bin[level] = seed.roll(level)

# Create hypervectors_table: each feature index has an hypervector
indexes_hvs_bin = torch.zeros([FEATURES, DIMENSIONS], dtype=torch.uint8)

# New seed
for i in range(DIMENSIONS):
   seed[i] = 1 if seed[i] == 0 else 0

for i in range(FEATURES):
    indexes_hvs_bin[i] = seed.roll(i)

In [ ]:
# Encode train set

TRAIN_SET_SIZE = 500
#TRAIN_SET_SIZE = 5000
#TRAIN_SET_SIZE = train_samples.shape[0] # Full train set
encoded_train_set = []
encoded_train_set.append(torch.zeros([TRAIN_SET_SIZE, DIMENSIONS], dtype=torch.int8))
encoded_train_set.append(torch.zeros([TRAIN_SET_SIZE, 1], dtype=torch.int8))

encoded_train_set[1] = train_labels

for i, sample in enumerate(train_samples[:TRAIN_SET_SIZE]):
    encoded_train_set[0][i], _ = Encode(DIMENSIONS, sample, level_hvs_bin, indexes_hvs_bin)

#torch.save(encoded_train_set[0], "train_sz60000_D10000_train60k.pt")
#encoded_train_set[0] = torch.load("drive/MyDrive/ColabNotebooks/HDC/sparse/coding/try2/train_sz60000_D10000_train60k.pt", weights_only = True)

train_set = {
    'samples' : encoded_train_set[0],
    'labels' : encoded_train_set[1]
}

In [ ]:
# Encode test set

TEST_SET_SIZE = 500
#TEST_SET_SIZE = 5000
#TEST_SET_SIZE = test_samples.shape[0] # Full train set
encoded_test_set = []
encoded_test_set.append(torch.zeros([TEST_SET_SIZE, DIMENSIONS], dtype=torch.int8))
encoded_test_set.append(torch.zeros([TEST_SET_SIZE, 1], dtype=torch.int8))

encoded_test_set[1] = test_labels

for i, sample in enumerate(test_samples[:TEST_SET_SIZE]):
    encoded_test_set[0][i], _ = Encode(DIMENSIONS, sample, level_hvs_bin, indexes_hvs_bin)

#torch.save(encoded_test_set[0], "train_sz60000_D10000_train60k.pt")
#encoded_test_set[0] = torch.load("drive/MyDrive/ColabNotebooks/HDC/sparse/coding/try2/train_sz60000_D10000_train60k.pt", weights_only = True)

test_set = {
    'samples' : encoded_test_set[0],
    'labels' : encoded_test_set[1]

}

In [ ]:
# Create class hypervectors using different amounts of train samples

train_sizes = [TEST_SET_SIZE]

class_hvs = torch.zeros([len(train_sizes), CLASSES, DIMENSIONS], dtype=torch.int32)
class_hvs_bin = torch.zeros([len(train_sizes), CLASSES, DIMENSIONS], dtype=torch.int8)

for i, size in enumerate(train_sizes):

    encoded_samples = train_set['samples'][:size]
    labels = train_set['labels'][:size]

    # Create hypervector classes
    # Standard centroid-based classifier: this is the most basic version of the classifier, where it adds each sample’s hypervector to its corresponding class
    for sample, label in zip(encoded_samples, labels.tolist()):
        class_hvs[i][label] += sample

    # Number of added samples in each class
    class_count = torch.bincount(labels)

    # Binarization: majority vote threshold
    for cl in range(CLASSES):
        class_hvs_bin[i][cl] = torch.where(class_hvs[i][cl] > (class_count[cl] // 2), 1, 0)

    print(f'class_hvs[{i}]: {CLASSES} class hypervectors created from {train_sizes[i]} train samples')

class_hvs[0]: 26 class hypervectors created from 500 train samples


In [ ]:
def Tensor2Int(t):

    value = 0

    for i in range(len(t)):
        value += t[i].item() * 2 ** i

    return value

In [ ]:
delta_h = []

for j in range(DIMENSIONS):
    max = 0
    min = 1000000
    for k in class_hvs_bin[0]:
        if k[j] > max:
            max = k[j]

        if k[j] < min:
            min = k[j]

    delta_h.append(max - min)

delta_h = torch.tensor(delta_h)

In [ ]:
sample = train_samples[5]

hv_bin, hv = Encode(DIMENSIONS, sample, level_hvs_bin, indexes_hvs_bin)

top = 3700
top_idxs = delta_h.topk(top).indices

image_file = open(f'ISOLET_sample.txt', "w+")
for level in sample.flatten():
    image_file.write(f'{level.item():02x}\n')
image_file.close()

indexes_file = open(f'ISOLET_idxs.txt', "w+")
for idx in top_idxs:
    indexes_file.write(f'{idx:04x}\n')
indexes_file.close()

class_hvs_bin_t = class_hvs_bin[0].T

indexes_file = open(f'ISOLET_hvs.txt', "w+")
for idx in top_idxs:
    #indexes_file.write(f'{Tensor2Int(class_hvs_bin_t[idx]):08x}\n') # reconf
    indexes_file.write(f'{Tensor2Int(class_hvs_bin_t[idx]):07x}\n') # hardwired
indexes_file.close()

In [ ]:
similarity = torch.zeros([CLASSES])

for cl in range(CLASSES):
    similarity[cl] = 1 - distance.hamming(class_hvs_bin[0][cl][top_idxs], hv_bin[top_idxs])

print(similarity * len(top_idxs))


similarity = torch.zeros([CLASSES])

for cl in range(CLASSES):
    for i in top_idxs:
        similarity[cl] += 1 if class_hvs_bin[0][cl][i] == hv_bin[i] else 0

print(similarity, similarity.argmin())

tensor([1972.0001, 2188.0000, 2583.0000, 2247.0000, 2213.0000, 1967.0000,
        2237.0000, 2031.0000, 1926.9999, 2110.0000, 2001.0000, 1972.0001,
        1952.0000, 1973.9999, 2020.9999, 2192.0000, 2115.0000, 1941.0000,
        1922.0001, 2328.0000, 2024.0001, 2190.0000, 1964.0001, 1950.0000,
        1881.0000, 2440.0000])
tensor([1972., 2188., 2583., 2247., 2213., 1967., 2237., 2031., 1927., 2110.,
        2001., 1972., 1952., 1974., 2021., 2192., 2115., 1941., 1922., 2328.,
        2024., 2190., 1964., 1950., 1881., 2440.]) tensor(24)
